# ASG Airlines — Bronze to Silver: Cleaning & Standardization

**Stage:** Bronze → Silver (Medallion Architecture)
**Reads from:** `bronze` container · **Writes to:** `silver` container

## What this notebook does

Applies the cleaning rules that came directly out of the Bronze-layer data quality profiling (not assumed rules — every rule below maps to a specific issue found by inspecting the actual data):

| # | Table | Issue found in profiling | Rule applied |
|---|---|---|---|
| 1 | flights | 15 `flight_id`s were fully-identical duplicate rows (30 rows) | Drop duplicates, keep first occurrence |
| 2 | flights | 1 `flight_id` (`6F250`) appeared twice with genuinely conflicting route/time data | Quarantine both rows — can't determine which is correct |
| 3 | flights | Raw `duration` column had 3 inconsistent formats (`HH:MM:SS`, fractional-microsecond, a corrupted Excel-epoch value) | Drop the column; duration is recomputed from timestamps in a later notebook |
| 4 | flights | `airline` was null or literal `'UNKNOWN'` in ~7% of rows | Set to `'Unknown'`, add flag `airline_is_missing` |
| 5 | payments | `amount` had nulls and literal `'INVALID'` strings | Cast to numeric, set unparseable to null, add flag `amount_is_invalid` |
| 6 | bookings | `status` had nulls | Set to `'Unknown'`, add flag `status_is_missing` |
| 7 | passengers | `last_name` had nulls | Set to `'Unknown'`, add flag `last_name_is_missing` |

**Guiding principle throughout: flag-and-retain, never silently drop.** A row with one bad field usually still carries valid information in its other fields, so every rule above adds a boolean indicator column rather than deleting the row — the only exception is the exact-duplicate case (Rule 1, which is genuinely redundant data) and the quarantine case (Rule 2, where we can't safely keep either conflicting version).

## Step 1 — Storage configuration & read Bronze tables

In [1]:
import os
import datetime
import pandas as pd
import numpy as np
from deltalake import write_deltalake, DeltaTable

STORAGE_ACCOUNT_NAME = "stasgairlines01"
CONTAINER_BRONZE     = "bronze"
CONTAINER_SILVER     = "silver"

STORAGE_KEY = os.environ.get("ADLS_STORAGE_KEY", "")
if not STORAGE_KEY:
    try:
        import subprocess
        res = subprocess.run(
            ["az", "storage", "account", "keys", "list",
             "--account-name", STORAGE_ACCOUNT_NAME,
             "--resource-group", "rg-asg-airlines",
             "--query", "[0].value", "-o", "tsv"],
            capture_output=True, text=True, check=True
        )
        STORAGE_KEY = res.stdout.strip()
    except Exception:
        pass

if not STORAGE_KEY:
    raise RuntimeError("ADLS_STORAGE_KEY environment variable is not set and az CLI lookup failed.")

storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT_NAME,
    "azure_storage_access_key": STORAGE_KEY,
}

df_flights_bronze    = DeltaTable(f"az://{CONTAINER_BRONZE}/flights", storage_options=storage_options).to_pandas()
df_bookings_bronze   = DeltaTable(f"az://{CONTAINER_BRONZE}/bookings", storage_options=storage_options).to_pandas()
df_passengers_bronze = DeltaTable(f"az://{CONTAINER_BRONZE}/passengers", storage_options=storage_options).to_pandas()
df_payments_bronze   = DeltaTable(f"az://{CONTAINER_BRONZE}/payments", storage_options=storage_options).to_pandas()

print(f"Bronze flights   : {len(df_flights_bronze):,} rows")
print(f"Bronze bookings  : {len(df_bookings_bronze):,} rows")
print(f"Bronze passengers: {len(df_passengers_bronze):,} rows")
print(f"Bronze payments  : {len(df_payments_bronze):,} rows")

summary_report = []

Bronze flights   : 1,020 rows
Bronze bookings  : 1,000 rows
Bronze passengers: 1,039 rows
Bronze payments  : 1,000 rows


## Rule 1 — Drop fully-identical duplicate `flights` rows

Duplicate detection uses an **exact full-row match across all original source columns** (not just `flight_id`). This is the detail that lets Rule 1 (true duplicates) and Rule 2 (genuine data conflicts, next cell) be told apart correctly — a duplicate `flight_id` alone isn't enough evidence to drop a row, since two rows could share an ID but disagree on the actual flight details, which is a different problem entirely.

In [1]:
df_flights = df_flights_bronze.copy()
initial_flights_cnt = len(df_flights)

source_cols_flights = ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']
distinct_flights_df = df_flights.drop_duplicates(subset=source_cols_flights, keep='first')
dedup_dropped_cnt = len(df_flights) - len(distinct_flights_df)
df_flights = distinct_flights_df.copy()

summary_report.append({
    "Table": "flights", "Rule #": 1,
    "Rule Description": "Drop fully-identical duplicate rows",
    "Affected Count": dedup_dropped_cnt,
    "Action Taken": f"Dropped {dedup_dropped_cnt} identical duplicate rows (kept 1 each)"
})
print(f"Dropped {dedup_dropped_cnt} exact duplicate rows")

Dropped 15 exact duplicate rows


## Rule 2 — Quarantine `flight_id`s with conflicting data

For any `flight_id` that appears more than once, we check whether the *other* fields also match. If they don't, that's not a duplicate — it's two different, conflicting records sharing an ID, and there's no reliable way to know which one is correct. Rather than guessing, both rows are moved to a separate quarantine table (`silver/rejected_flights_id_conflicts`) with a reason and timestamp, and excluded from the main Silver `flights` table.

In [1]:
flight_id_counts = df_flights['flight_id'].value_counts()
potential_conflicts = flight_id_counts[flight_id_counts > 1].index

conflicting_flight_ids = []
for fid in potential_conflicts:
    sub = df_flights[df_flights['flight_id'] == fid][source_cols_flights]
    if len(sub.drop_duplicates()) > 1:
        conflicting_flight_ids.append(fid)

quarantine_mask = df_flights['flight_id'].isin(conflicting_flight_ids)
df_quarantine = df_flights[quarantine_mask].copy()
df_flights = df_flights[~quarantine_mask].copy()

df_quarantine['rejection_reason'] = "Conflicting flight_id details across rows"
df_quarantine['quarantine_timestamp'] = datetime.datetime.now(datetime.timezone.utc)

quarantine_cnt = len(df_quarantine)
summary_report.append({
    "Table": "flights", "Rule #": 2,
    "Rule Description": "Quarantine conflicting flight_id records",
    "Affected Count": quarantine_cnt,
    "Action Taken": f"Quarantined {quarantine_cnt} rows (flight_ids: {conflicting_flight_ids}) to silver/rejected_flights_id_conflicts"
})
print(f"Quarantined {quarantine_cnt} rows: {conflicting_flight_ids}")

Quarantined 2 rows: ['6F250']


## Rule 3 — Drop the raw `duration` column

Profiling found this column mixed three formats (`HH:MM:SS`, fractional-microsecond timestamps, and one corrupted Excel-epoch value). Rather than write fragile parsing logic for three inconsistent formats, we drop the column outright — duration is recomputed cleanly from `departure_time`/`arrival_time` in the next notebook (`03_recompute_flight_duration.ipynb`), which is a more reliable source of truth than a pre-computed field we can't fully trust.

In [1]:
if 'duration' in df_flights.columns:
    df_flights = df_flights.drop(columns=['duration'])

summary_report.append({
    "Table": "flights", "Rule #": 3,
    "Rule Description": "Drop raw duration column",
    "Affected Count": initial_flights_cnt,
    "Action Taken": "Dropped raw 'duration' column (recomputed in the next notebook)"
})
print("Dropped raw duration column")

Dropped raw duration column


## Rule 4 — Standardize missing `airline` values

Missing values in this dataset showed up two ways: true nulls, and the literal string `'UNKNOWN'` — both mean the same thing and are treated identically here.

In [1]:
airline_null_mask = (
    df_flights['airline'].isna()
    | (df_flights['airline'].astype(str).str.strip() == '')
    | (df_flights['airline'] == 'UNKNOWN')
    | (df_flights['airline'] == 'Unknown')
)
airline_missing_cnt = airline_null_mask.sum()

df_flights['airline_is_missing'] = airline_null_mask
df_flights.loc[airline_null_mask, 'airline'] = "Unknown"

summary_report.append({
    "Table": "flights", "Rule #": 4,
    "Rule Description": "Standardize missing airline",
    "Affected Count": int(airline_missing_cnt),
    "Action Taken": f"Set {airline_missing_cnt} null/UNKNOWN airlines to 'Unknown', flag airline_is_missing = True"
})
print(f"Flagged {airline_missing_cnt} missing airline values")

Flagged 67 missing airline values


## Rule 5 — Cast `payments.amount` to numeric

We deliberately do **not** impute a fabricated amount for missing/invalid payments — inventing a financial figure would misrepresent revenue. Instead, unparseable values become `null` with a flag column, so they can be explicitly excluded from any amount-based KPI later rather than silently pulling the average down (or up).

In [1]:
df_payments = df_payments_bronze.copy()

numeric_amounts = pd.to_numeric(df_payments['amount'], errors='coerce')
amount_invalid_mask = numeric_amounts.isna()
amount_invalid_cnt = int(amount_invalid_mask.sum())

df_payments['amount_is_invalid'] = amount_invalid_mask
df_payments['amount'] = numeric_amounts

summary_report.append({
    "Table": "payments", "Rule #": 5,
    "Rule Description": "Cast amount to numeric & flag invalid/nulls",
    "Affected Count": amount_invalid_cnt,
    "Action Taken": f"Cast amount to float. Set {amount_invalid_cnt} invalid/null amounts to null, flag amount_is_invalid = True"
})
print(f"Flagged {amount_invalid_cnt} invalid/missing payment amounts")

Flagged 78 invalid/missing payment amounts


## Rule 6 — Standardize missing `bookings.status`

In [1]:
df_bookings = df_bookings_bronze.copy()

status_null_mask = df_bookings['status'].isna() | (df_bookings['status'].astype(str).str.strip() == '')
status_missing_cnt = int(status_null_mask.sum())

df_bookings['status_is_missing'] = status_null_mask
df_bookings.loc[status_null_mask, 'status'] = "Unknown"

summary_report.append({
    "Table": "bookings", "Rule #": 6,
    "Rule Description": "Standardize missing booking status",
    "Affected Count": status_missing_cnt,
    "Action Taken": f"Set {status_missing_cnt} null status values to 'Unknown', flag status_is_missing = True"
})
print(f"Flagged {status_missing_cnt} missing booking statuses")

Flagged 45 missing booking statuses


## Rule 7 — Standardize missing `passengers.last_name`

In [1]:
df_passengers = df_passengers_bronze.copy()

lname_null_mask = df_passengers['last_name'].isna() | (df_passengers['last_name'].astype(str).str.strip() == '')
lname_missing_cnt = int(lname_null_mask.sum())

df_passengers['last_name_is_missing'] = lname_null_mask
df_passengers.loc[lname_null_mask, 'last_name'] = "Unknown"

summary_report.append({
    "Table": "passengers", "Rule #": 7,
    "Rule Description": "Standardize missing last_name",
    "Affected Count": lname_missing_cnt,
    "Action Taken": f"Set {lname_missing_cnt} null last_names to 'Unknown', flag last_name_is_missing = True"
})
print(f"Flagged {lname_missing_cnt} missing last names")

Flagged 10 missing last names


## Cleaning summary report

Before writing anything, print a consolidated before/after report — this is the main way an unexpected row-count change would get caught immediately rather than propagating silently into Silver.

In [1]:
print("=" * 80)
print("ASG AIRLINES — SILVER CLEANING & STANDARDIZATION REPORT")
print("=" * 80)

summary_df = pd.DataFrame(summary_report)
for idx, row in summary_df.iterrows():
    print(f"\n[Rule {row['Rule #']}] {row['Table'].upper()}: {row['Rule Description']}")
    print(f"         Affected Rows : {row['Affected Count']}")
    print(f"         Action Taken  : {row['Action Taken']}")

print("\nBEFORE vs AFTER ROW COUNTS:")
print(f"  flights    : Bronze = {len(df_flights_bronze):>5} -> Silver = {len(df_flights):>5}  (Dropped {dedup_dropped_cnt} dups, Quarantined {quarantine_cnt})")
print(f"  bookings   : Bronze = {len(df_bookings_bronze):>5} -> Silver = {len(df_bookings):>5}")
print(f"  passengers : Bronze = {len(df_passengers_bronze):>5} -> Silver = {len(df_passengers):>5}")
print(f"  payments   : Bronze = {len(df_payments_bronze):>5} -> Silver = {len(df_payments):>5}")
print(f"  quarantine : Rejected Flights ID Conflicts = {len(df_quarantine):>5} rows")

ASG AIRLINES — SILVER CLEANING & STANDARDIZATION REPORT

[Rule 1] FLIGHTS: Drop fully-identical duplicate rows
         Affected Rows : 15
         Action Taken  : Dropped 15 identical duplicate rows (kept 1 each)

[Rule 2] FLIGHTS: Quarantine conflicting flight_id records
         Affected Rows : 2
         Action Taken  : Quarantined 2 rows (flight_ids: ['6F250']) to silver/rejected_flights_id_conflicts

[Rule 3] FLIGHTS: Drop raw duration column
         Affected Rows : 1020
         Action Taken  : Dropped raw 'duration' column (recomputed in the next notebook)

[Rule 4] FLIGHTS: Standardize missing airline
         Affected Rows : 67
         Action Taken  : Set 67 null/UNKNOWN airlines to 'Unknown', flag airline_is_missing = True

[Rule 5] PAYMENTS: Cast amount to numeric & flag invalid/nulls
         Affected Rows : 78
         Action Taken  : Cast amount to float. Set 78 invalid/null amounts to null, flag amount_is_invalid = True

[Rule 6] BOOKINGS: Standardize missing booking 

## Write Silver Delta tables

Also saves a local copy of the quarantine table (`data/rejected/flights_id_conflicts.parquet`) for easy manual inspection without needing a storage client.

In [1]:
silver_tables = {
    "flights": df_flights,
    "bookings": df_bookings,
    "passengers": df_passengers,
    "payments": df_payments,
    "rejected_flights_id_conflicts": df_quarantine
}

for name, df in silver_tables.items():
    silver_uri = f"az://{CONTAINER_SILVER}/{name}"
    write_deltalake(silver_uri, df, mode="overwrite", schema_mode="overwrite", storage_options=storage_options)
    print(f"Wrote {len(df):,} rows -> {silver_uri}")

local_quarantine_dir = os.path.join("data", "rejected")
os.makedirs(local_quarantine_dir, exist_ok=True)
local_quarantine_path = os.path.join(local_quarantine_dir, "flights_id_conflicts.parquet")
df_quarantine.to_parquet(local_quarantine_path, index=False)
print(f"Saved local copy of quarantine records -> {local_quarantine_path}")

Wrote 1,003 rows -> az://silver/flights
Wrote 1,000 rows -> az://silver/bookings
Wrote 1,039 rows -> az://silver/passengers
Wrote 1,000 rows -> az://silver/payments
Wrote 2 rows -> az://silver/rejected_flights_id_conflicts
Saved local copy of quarantine records -> data/rejected/flights_id_conflicts.parquet


## Verify Silver tables by reading them back

In [1]:
for name in silver_tables.keys():
    silver_uri = f"az://{CONTAINER_SILVER}/{name}"
    dt = DeltaTable(silver_uri, storage_options=storage_options)
    v_df = dt.to_pandas()
    print(f"[Verified] silver/{name:<30}: {len(v_df):,} rows | Schema fields: {len(dt.schema().fields)}")

print("\nTransformation to Silver layer complete.")

[Verified] silver/flights                      : 1,003 rows | Schema fields: 10
[Verified] silver/bookings                     : 1,000 rows | Schema fields: 12
[Verified] silver/passengers                   : 1,039 rows | Schema fields: 12
[Verified] silver/payments                     : 1,000 rows | Schema fields: 7
[Verified] silver/rejected_flights_id_conflicts: 2 rows | Schema fields: 12

Transformation to Silver layer complete.
